# LeetCode 994 — Rotting Oranges bằng Python

Link đề: https://leetcode.com/problems/rotting-oranges/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (số phút tối thiểu, hoặc `-1`). Điểm cốt lõi: các quả cam thối lan **đồng thời** theo từng phút ⇒ **BFS nhiều nguồn** (multi-source BFS) theo từng **tầng**; kết quả là số tầng nên đếm đúng (tránh lệch 1) và phải kiểm tra còn cam tươi không với được không. Oracle độc lập: **mô phỏng từng phút** đúng nguyên văn đề.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `m × n` gồm `0` (trống), `1` (cam tươi), `2` (cam thối); mỗi phút mọi cam tươi **kề 4 hướng** với cam thối bị thối; tìm số phút tối thiểu tới khi hết cam tươi, hoặc `-1`.
2. Nắm công cụ: `collections.deque`, BFS theo tầng, đếm số cam tươi còn lại, mảng hướng đi `(0,1),(1,0),(0,-1),(-1,0)`.
3. Đi từ **mô phỏng từng phút** → **BFS nhiều nguồn theo tầng** → **BFS gắn nhãn thời gian** → **BFS từ từng cam thối rồi lấy min** → **lan truyền nới lỏng đến khi ổn định**.
4. Hiểu mấu chốt: thời điểm cam tại ô `c` thối = **khoảng cách ngắn nhất** (qua các ô không trống, 4 hướng) từ ô `c` tới **cam thối gần nhất**; đáp án = **max** các thời điểm đó, hoặc `-1` nếu có cam tươi không với tới.
5. Tránh lỗi kinh điển: cộng thừa một phút (đếm tầng cuối không còn cam tươi để lây), quên trả `-1` khi còn cam tươi, dùng 8 hướng, chỉ BFS từ **một** cam thối, lây liền trong cùng một phút (sửa lưới tại chỗ khi duyệt), trả `-1` sai khi không có cam tươi nào (đáp án phải là `0`).

## 2. Đề bài & ví dụ

Cho lưới `grid` kích thước `m × n`, mỗi ô là:

- `0`: ô trống;
- `1`: cam tươi;
- `2`: cam thối.

Mỗi phút, mọi cam tươi **kề 4 hướng** (trên, dưới, trái, phải) với một cam thối sẽ bị thối.

Trả về **số phút tối thiểu** để không còn ô nào có cam tươi. Nếu không thể, trả về `-1`.

**Ví dụ 1:** `grid = [[2,1,1],[1,1,0],[0,1,1]]` → `4`.

**Ví dụ 2:** `grid = [[2,1,1],[0,1,1],[1,0,1]]` → `-1` (cam ở góc dưới trái không bao giờ thối vì chỉ lan 4 hướng).

**Ví dụ 3:** `grid = [[0,2]]` → `0` (không có cam tươi ngay từ đầu).

**Ràng buộc:** `1 <= m, n <= 10`; `grid[i][j]` là `0`, `1` hoặc `2`.

In [1]:
import itertools
import random
import time
from collections import deque
from typing import List

ex = [([[2, 1, 1], [1, 1, 0], [0, 1, 1]], 4), ([[2, 1, 1], [0, 1, 1], [1, 0, 1]], -1), ([[0, 2]], 0)]
for g, want in ex:
    print(f"grid = {g} -> {want}")

grid = [[2, 1, 1], [1, 1, 0], [0, 1, 1]] -> 4
grid = [[2, 1, 1], [0, 1, 1], [1, 0, 1]] -> -1
grid = [[0, 2]] -> 0


## 3. Kiến thức nền

### 3.1 Hướng đi và biên

Bốn hướng: `(0, 1), (1, 0), (0, -1), (-1, 0)`. Với ô `(r, c)`, ô kề `(r + dr, c + dc)` hợp lệ khi `0 ≤ r + dr < m` và `0 ≤ c + dc < n`.

In [2]:
DIRS = ((0, 1), (1, 0), (0, -1), (-1, 0))


def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        rr, cc = r + dr, c + dc
        if 0 <= rr < m and 0 <= cc < n:
            yield rr, cc


print(list(neighbors(0, 0, 3, 3)), "|", list(neighbors(1, 1, 3, 3)))

[(0, 1), (1, 0)] | [(1, 2), (2, 1), (1, 0), (0, 1)]


### 3.2 Một phút lan truyền

Trong **một** phút, ta xét trạng thái đầu phút: mọi cam tươi kề một cam thối (ở trạng thái **đầu phút**) bị thối. Cam vừa thối **chưa** lây cho cam khác trong cùng phút ấy. Đây là chi tiết dễ sai khi duyệt và sửa lưới tại chỗ.

In [3]:
def one_minute(grid):
    """Trả về lưới sau đúng một phút (dựa trên trạng thái ĐẦU phút)."""
    m, n = len(grid), len(grid[0])
    new = [row[:] for row in grid]
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1 and any(grid[rr][cc] == 2 for rr, cc in neighbors(r, c, m, n)):
                new[r][c] = 2
    return new


g = [[2, 1, 1], [1, 1, 0], [0, 1, 1]]
for minute in range(1, 5):
    g = one_minute(g)
    print(minute, g)

1 [[2, 2, 1], [2, 1, 0], [0, 1, 1]]
2 [[2, 2, 2], [2, 2, 0], [0, 1, 1]]
3 [[2, 2, 2], [2, 2, 0], [0, 2, 1]]
4 [[2, 2, 2], [2, 2, 0], [0, 2, 2]]


### 3.3 BFS nhiều nguồn

Đưa **mọi** cam thối ban đầu vào hàng đợi cùng lúc (khoảng cách `0`) rồi BFS: các ô được thăm ở tầng `k` bị thối đúng ở phút `k`. Đây là "BFS nhiều nguồn" (multi-source BFS).

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 theo thời gian:

| phút | các ô vừa thối | lưới sau phút đó |
|---|---|---|
| 0 | `(0,0)` (ban đầu) | `[[2,1,1],[1,1,0],[0,1,1]]` |
| 1 | `(0,1)`, `(1,0)` | `[[2,2,1],[2,1,0],[0,1,1]]` |
| 2 | `(0,2)`, `(1,1)` | `[[2,2,2],[2,2,0],[0,1,1]]` |
| 3 | `(2,1)` | `[[2,2,2],[2,2,0],[0,2,1]]` |
| 4 | `(2,2)` | `[[2,2,2],[2,2,0],[0,2,2]]` |

Hết cam tươi sau `4` phút. Mỗi "đợt" chính là **một tầng BFS**. Ví dụ 2: cam ở `(2,0)` bị chặn bởi ô trống `(1,0)` và `(2,1)`, nên không bao giờ thối ⇒ `-1`.

## 5. Brute force — mô phỏng từng phút

**Ý tưởng:** đúng nguyên văn đề: lặp `one_minute` cho đến khi lưới không còn thay đổi; đếm số phút **có thay đổi**. Cuối cùng nếu còn cam tươi ⇒ `-1`. Mỗi vòng `O(mn)`, tối đa `mn` vòng ⇒ `O((mn)²)` — vẫn rất nhỏ với `m, n ≤ 10`. Đây là **oracle** độc lập.

In [4]:
def rotting_simulate(grid):
    g = [row[:] for row in grid]
    minutes = 0
    while True:
        new = one_minute(g)
        if new == g:                                   # không còn gì thay đổi
            break
        g = new
        minutes += 1
    return -1 if any(1 in row for row in g) else minutes


for g, want in ex:
    assert rotting_simulate(g) == want
print([rotting_simulate(g) for g, _ in ex])

[4, -1, 0]


In [5]:
# Code nộp LeetCode cho Brute force (mô phỏng từng phút; vẫn qua vì lưới <= 10x10)
class SolutionSimulate:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        minutes = 0
        while True:
            new = [row[:] for row in g]
            for r in range(m):
                for c in range(n):
                    if g[r][c] == 1:
                        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                            rr, cc = r + dr, c + dc
                            if 0 <= rr < m and 0 <= cc < n and g[rr][cc] == 2:
                                new[r][c] = 2
                                break
            if new == g:
                break
            g = new
            minutes += 1
        return -1 if any(1 in row for row in g) else minutes

**Độ phức tạp:** `O((mn)²)` thời gian (`≤ 10^4` với lưới `10 × 10`), `O(mn)` bộ nhớ.

**Nút thắt:** mỗi phút ta quét lại cả lưới dù chỉ các ô **mới** thối mới có thể lây tiếp. BFS chỉ xử lý "mép" lan truyền.

## 6. Bước đột phá

Coi lưới là đồ thị (ô không trống là đỉnh, kề 4 hướng). Thời điểm một ô cam tươi thối = khoảng cách BFS ngắn nhất từ tập cam thối ban đầu. BFS nhiều nguồn theo tầng: mỗi tầng một phút; đếm số cam tươi còn lại để biết có `-1` không. `O(mn)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** BFS nhiều nguồn theo tầng (đáp án chính), **Cách 3** BFS gắn nhãn thời gian, **Cách 4** BFS từ từng cam thối rồi lấy min, **Cách 5** lan truyền nới lỏng đến khi ổn định.

### Cách 2 — BFS nhiều nguồn theo tầng

**Giải thích từng đoạn:**

1. Duyệt lưới: đưa mọi ô `2` vào hàng đợi; đếm `fresh` = số ô `1`.
2. `minutes = 0`. Khi hàng đợi còn phần tử **và** `fresh > 0`: xử lý **đúng một tầng** (`len(queue)` phần tử hiện tại); mỗi cam tươi kề được đánh dấu thối (`grid = 2`), `fresh -= 1`, đưa vào hàng đợi; sau tầng, `minutes += 1`.
3. Kết quả: `minutes` nếu `fresh == 0`, ngược lại `-1`.

Điều kiện `fresh > 0` ở vòng ngoài tránh **cộng thừa** một phút cho tầng cuối không lây thêm được.

In [6]:
def rotting_bfs(grid):
    g = [row[:] for row in grid]                      # không sửa đầu vào
    m, n = len(g), len(g[0])
    queue, fresh = deque(), 0
    for r in range(m):
        for c in range(n):
            if g[r][c] == 2:
                queue.append((r, c))
            elif g[r][c] == 1:
                fresh += 1
    minutes = 0
    while queue and fresh:
        for _ in range(len(queue)):                    # đúng một tầng = một phút
            r, c = queue.popleft()
            for rr, cc in neighbors(r, c, m, n):
                if g[rr][cc] == 1:
                    g[rr][cc] = 2                      # thối ở phút này; chỉ lây tiếp từ phút sau (tầng sau)
                    fresh -= 1
                    queue.append((rr, cc))
        minutes += 1
    return minutes if fresh == 0 else -1


for g, want in ex:
    assert rotting_bfs(g) == want
print([rotting_bfs(g) for g, _ in ex])

[4, -1, 0]


In [7]:
# Code nộp LeetCode cho Cách 2
from collections import deque


class SolutionBFS:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        queue, fresh = deque(), 0
        for r in range(m):
            for c in range(n):
                if g[r][c] == 2:
                    queue.append((r, c))
                elif g[r][c] == 1:
                    fresh += 1
        minutes = 0
        while queue and fresh:
            for _ in range(len(queue)):
                r, c = queue.popleft()
                for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < m and 0 <= cc < n and g[rr][cc] == 1:
                        g[rr][cc] = 2
                        fresh -= 1
                        queue.append((rr, cc))
            minutes += 1
        return minutes if fresh == 0 else -1

**Độ phức tạp:** `O(mn)` thời gian (mỗi ô vào hàng đợi tối đa một lần), `O(mn)` bộ nhớ.

### Cách 3 — BFS gắn nhãn thời gian

**Ý tưởng:** hàng đợi chứa `(r, c, t)` với `t` là phút mà ô đó thối. Không cần vòng lặp theo tầng; đáp án = `t` lớn nhất gặp được. Đếm `fresh` để quyết định `-1`.

In [8]:
def rotting_timestamp(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue = deque((r, c, 0) for r in range(m) for c in range(n) if g[r][c] == 2)
    fresh = sum(row.count(1) for row in g)
    last = 0
    while queue:
        r, c, t = queue.popleft()
        last = max(last, t)
        for rr, cc in neighbors(r, c, m, n):
            if g[rr][cc] == 1:
                g[rr][cc] = 2
                fresh -= 1
                queue.append((rr, cc, t + 1))
    return last if fresh == 0 else -1


for g, want in ex:
    assert rotting_timestamp(g) == want
print([rotting_timestamp(g) for g, _ in ex])

[4, -1, 0]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionTimestamp:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        g = [row[:] for row in grid]
        queue = deque((r, c, 0) for r in range(m) for c in range(n) if g[r][c] == 2)
        fresh = sum(row.count(1) for row in g)
        last = 0
        while queue:
            r, c, t = queue.popleft()
            last = max(last, t)
            for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                rr, cc = r + dr, c + dc
                if 0 <= rr < m and 0 <= cc < n and g[rr][cc] == 1:
                    g[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc, t + 1))
        return last if fresh == 0 else -1

**Độ phức tạp:** `O(mn)` thời gian, `O(mn)` bộ nhớ.

### Cách 4 — BFS từ từng cam thối rồi lấy min

**Ý tưởng:** với mỗi cam thối ban đầu chạy một BFS **riêng** (qua các ô không trống) để có khoảng cách tới mọi ô; thời điểm ô `c` thối = **min** các khoảng cách. Đáp án = max trên các cam tươi (nếu có ô vô cực ⇒ `-1`). Chi phí `O(R · mn)` (`R` số cam thối) — đủ cho lưới nhỏ và cho thấy trực tiếp công thức "max của min khoảng cách".

In [10]:
INF = float("inf")


def bfs_from(grid, src):
    m, n = len(grid), len(grid[0])
    dist = [[INF] * n for _ in range(m)]
    dist[src[0]][src[1]] = 0
    queue = deque([src])
    while queue:
        r, c = queue.popleft()
        for rr, cc in neighbors(r, c, m, n):
            if grid[rr][cc] != 0 and dist[rr][cc] == INF:
                dist[rr][cc] = dist[r][c] + 1
                queue.append((rr, cc))
    return dist


def rotting_per_source(grid):
    m, n = len(grid), len(grid[0])
    sources = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == 2]
    best = [[INF] * n for _ in range(m)]
    for src in sources:
        d = bfs_from(grid, src)
        for r in range(m):
            for c in range(n):
                best[r][c] = min(best[r][c], d[r][c])
    worst = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                if best[r][c] == INF:
                    return -1
                worst = max(worst, best[r][c])
    return worst


for g, want in ex:
    assert rotting_per_source(g) == want
print([rotting_per_source(g) for g, _ in ex])

[4, -1, 0]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionPerSource:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        INF = float("inf")
        best = [[INF] * n for _ in range(m)]
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] != 2:
                    continue
                dist = {(sr, sc): 0}
                queue = deque([(sr, sc)])
                while queue:
                    r, c = queue.popleft()
                    best[r][c] = min(best[r][c], dist[(r, c)])
                    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                        rr, cc = r + dr, c + dc
                        if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] != 0 and (rr, cc) not in dist:
                            dist[(rr, cc)] = dist[(r, c)] + 1
                            queue.append((rr, cc))
        worst = 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 1:
                    if best[r][c] == INF:
                        return -1
                    worst = max(worst, best[r][c])
        return worst

**Độ phức tạp:** `O(R · mn)` thời gian (`R ≤ mn`), `O(mn)` bộ nhớ.

### Cách 5 — Lan truyền nới lỏng đến khi ổn định

**Ý tưởng:** `t[r][c]` = thời điểm thối (`0` cho cam thối ban đầu, `∞` cho cam tươi, bỏ qua ô trống). Lặp: với mỗi cam tươi, `t[r][c] = min(t[r][c], 1 + min(t của các ô kề không trống))`; dừng khi không còn thay đổi (kiểu Bellman–Ford). Đáp án = max `t` trên cam tươi (hoặc `-1` nếu còn `∞`).

In [12]:
def rotting_relax(grid):
    m, n = len(grid), len(grid[0])
    t = [[0 if grid[r][c] == 2 else INF for c in range(n)] for r in range(m)]
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 1:
                    best = min((t[rr][cc] for rr, cc in neighbors(r, c, m, n) if grid[rr][cc] != 0), default=INF)
                    if best + 1 < t[r][c]:
                        t[r][c] = best + 1
                        changed = True
    worst = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                if t[r][c] == INF:
                    return -1
                worst = max(worst, t[r][c])
    return worst


for g, want in ex:
    assert rotting_relax(g) == want
print([rotting_relax(g) for g, _ in ex])

[4, -1, 0]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionRelax:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        INF = float("inf")
        t = [[0 if grid[r][c] == 2 else INF for c in range(n)] for r in range(m)]
        changed = True
        while changed:
            changed = False
            for r in range(m):
                for c in range(n):
                    if grid[r][c] == 1:
                        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                            rr, cc = r + dr, c + dc
                            if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] != 0 and t[rr][cc] + 1 < t[r][c]:
                                t[r][c] = t[rr][cc] + 1
                                changed = True
        worst = 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 1:
                    if t[r][c] == INF:
                        return -1
                    worst = max(worst, t[r][c])
        return worst

**Độ phức tạp:** mỗi vòng `O(mn)`, số vòng `≤ mn` (đường ngắn nhất tối đa `mn` bước) ⇒ `O((mn)²)` xấu nhất; `O(mn)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `grid = [[2,1,1],[1,1,0],[0,1,1]]`:

| tầng (phút) | hàng đợi đầu tầng | các ô mới thối | `fresh` sau tầng |
|---|---|---|---|
| (khởi tạo) | `[(0,0)]` | — | 6 |
| 1 | `[(0,0)]` | `(0,1)`, `(1,0)` | 4 |
| 2 | `[(0,1),(1,0)]` | `(0,2)`, `(1,1)` | 2 |
| 3 | `[(0,2),(1,1)]` | `(2,1)` | 1 |
| 4 | `[(2,1)]` | `(2,2)` | 0 |

`fresh = 0` ⇒ trả `minutes = 4`. Hàm trace in từng tầng:

In [14]:
def trace_bfs(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if g[r][c] == 2)
    fresh = sum(row.count(1) for row in g)
    print(f"khởi tạo: hàng đợi {list(queue)}, fresh = {fresh}")
    minutes = 0
    while queue and fresh:
        level = list(queue)
        new = []
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for rr, cc in neighbors(r, c, m, n):
                if g[rr][cc] == 1:
                    g[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc))
                    new.append((rr, cc))
        minutes += 1
        print(f"  phút {minutes}: hàng đợi đầu tầng {level} -> mới thối {new}, fresh = {fresh}")
    res = minutes if fresh == 0 else -1
    print("=>", res)
    return res


assert trace_bfs([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert trace_bfs([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1

khởi tạo: hàng đợi [(0, 0)], fresh = 6
  phút 1: hàng đợi đầu tầng [(0, 0)] -> mới thối [(0, 1), (1, 0)], fresh = 4
  phút 2: hàng đợi đầu tầng [(0, 1), (1, 0)] -> mới thối [(0, 2), (1, 1)], fresh = 2
  phút 3: hàng đợi đầu tầng [(0, 2), (1, 1)] -> mới thối [(2, 1)], fresh = 1
  phút 4: hàng đợi đầu tầng [(2, 1)] -> mới thối [(2, 2)], fresh = 0
=> 4
khởi tạo: hàng đợi [(0, 0)], fresh = 6
  phút 1: hàng đợi đầu tầng [(0, 0)] -> mới thối [(0, 1)], fresh = 5
  phút 2: hàng đợi đầu tầng [(0, 1)] -> mới thối [(0, 2), (1, 1)], fresh = 3
  phút 3: hàng đợi đầu tầng [(0, 2), (1, 1)] -> mới thối [(1, 2)], fresh = 2
  phút 4: hàng đợi đầu tầng [(1, 2)] -> mới thối [(2, 2)], fresh = 1
  phút 5: hàng đợi đầu tầng [(2, 2)] -> mới thối [], fresh = 1
=> -1


**Trace trường hợp không có cam tươi:** `[[0, 2]]`: `fresh = 0` ngay từ đầu nên vòng lặp không chạy ⇒ `0` (không `-1`, không `1`).

In [15]:
assert rotting_bfs([[0, 2]]) == 0 and rotting_bfs([[0]]) == 0 and rotting_bfs([[2]]) == 0 and rotting_bfs([[1]]) == -1
print(rotting_bfs([[0, 2]]), rotting_bfs([[0]]), rotting_bfs([[2]]), rotting_bfs([[1]]))

0 0 0 -1


## 9. Code nộp LeetCode

Chọn **Cách 2** (BFS nhiều nguồn theo tầng). Giữ nguyên `class Solution`, `orangesRotting(self, grid)`:

In [16]:
class Solution:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        queue, fresh = deque(), 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 2:
                    queue.append((r, c))                  # mọi cam thối là nguồn BFS (khoảng cách 0)
                elif grid[r][c] == 1:
                    fresh += 1                            # đếm cam tươi để biết còn sót không
        minutes = 0
        while queue and fresh:                            # dừng ngay khi hết cam tươi (tránh cộng thừa)
            for _ in range(len(queue)):                   # đúng một tầng = một phút
                r, c = queue.popleft()
                for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] == 1:
                        grid[rr][cc] = 2                  # thối; chỉ lây tiếp ở tầng sau
                        fresh -= 1
                        queue.append((rr, cc))
            minutes += 1
        return minutes if fresh == 0 else -1


print([Solution().orangesRotting([row[:] for row in g]) for g, _ in ex])
assert all(Solution().orangesRotting([row[:] for row in g]) == w for g, w in ex)

[4, -1, 0]


## 10. Vì sao đúng

**(a) Mô hình đồ thị.** Xét đồ thị `G` với đỉnh là các ô **không trống** và cạnh giữa hai ô kề 4 hướng. Một cam tươi ở ô `x` thối ở phút `t` nếu `t − 1` là thời điểm sớm nhất mà một ô kề của `x` thối (tức tồn tại ô kề thối ở đầu phút `t`). Quy nạp theo `t`: tập ô thối ở cuối phút `t` = tập ô có khoảng cách `≤ t` tới tập cam thối ban đầu `S` trong `G`. (Cơ sở `t = 0`: khoảng cách `0` là `S`. Bước: ô `x` thối ở cuối phút `t` ⇔ đã thối hoặc có ô kề thối ở cuối phút `t − 1` ⇔ `d(x, S) ≤ t`.) Vậy thời điểm thối của `x` đúng bằng `d(x, S)`.

**(b) Đáp án.** Số phút tối thiểu để không còn cam tươi = `max` của `d(x, S)` trên mọi cam tươi `x` (nếu có ô `d = ∞` thì không bao giờ hết ⇒ `-1`); không có cam tươi thì `0`.

**(c) BFS nhiều nguồn theo tầng.** Hàng đợi chứa các ô ở tầng `k` (khoảng cách `k` tới `S`), xử lý tầng một lần ⇒ mọi ô tầng `k + 1` được thêm đúng một lần (đánh dấu `grid = 2` ngay khi thêm). Số tầng có cam mới = `max d`. Vòng lặp dừng khi `fresh = 0`: ngay sau tầng cuối cùng làm thối cam cuối, `minutes` bằng đúng `max d`; không đếm thêm tầng "rỗng". Nếu hàng đợi cạn mà `fresh > 0` thì còn ô không với tới ⇒ `-1`.

**(d) Cách 3** là cùng BFS với nhãn thời gian: nhãn của ô = khoảng cách; đáp án = nhãn lớn nhất (các ô thối ban đầu có nhãn `0`, nhãn lớn nhất chỉ tính các ô thối sau đó).

**(e) Cách 4.** `d(x, S) = min_{s ∈ S} d(x, s)`; BFS riêng từ mỗi `s` cho `d(·, s)` (cùng đồ thị `G`, ô trống bị loại).

**(f) Cách 5.** Quan hệ `t(x) = 1 + min_{y ~ x} t(y)` (với `x` tươi) là phương trình Bellman cho đường đi ngắn nhất với trọng số `1`; lặp nới lỏng hội tụ về nghiệm bé nhất (khoảng cách ngắn nhất) sau tối đa `|V|` vòng.

**(g) Brute force đúng.** `one_minute` thực hiện đúng định nghĩa một phút dựa trên trạng thái đầu phút; lặp tới khi không đổi cho số phút chính xác (mỗi vòng có thay đổi là một phút, vì cam thối chỉ tăng).

**(h) Kiểm chứng bằng máy.** Mọi cách khớp oracle mô phỏng trên **mọi** lưới `1 × n` (`n ≤ 8`), `2 × n` (`n ≤ 5`), `3 × 3` và một mẫu lớn các lưới `3 × 4`, `4 × 4`, ... tới `10 × 10` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng phút | `O((mn)²)` | `O(mn)` | oracle, đúng nguyên văn |
| 2. BFS nhiều nguồn theo tầng | `O(mn)` | `O(mn)` | **đáp án chính** |
| 3. BFS gắn nhãn thời gian | `O(mn)` | `O(mn)` | không cần vòng theo tầng |
| 4. BFS từng cam thối + min | `O(R·mn)` | `O(mn)` | công thức "max của min" |
| 5. Nới lỏng đến ổn định | `O((mn)²)` | `O(mn)` | Bellman–Ford trên lưới |

Đo thực tế (lưới `10 × 10` ngẫu nhiên):

In [17]:
def random_grid(rng, m, n, p_empty=0.2, p_rotten=0.1):
    g = []
    for _ in range(m):
        row = []
        for _ in range(n):
            x = rng.random()
            row.append(0 if x < p_empty else (2 if x < p_empty + p_rotten else 1))
        g.append(row)
    return g


def bench(fn, grid, repeat=200):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(grid)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(994)
big = random_grid(rng, 10, 10, 0.1, 0.03)
for name, fn in [("simulate", rotting_simulate), ("bfs", rotting_bfs), ("timestamp", rotting_timestamp), ("per_source", rotting_per_source), ("relax", rotting_relax)]:
    print(f"{name:11s}: {bench(fn, big) * 1e6:8.1f} us -> {fn(big)}")

simulate   :    231.2 us -> 7


bfs        :     73.3 us -> 7


timestamp  :     78.1 us -> 7


per_source :    424.9 us -> 7


relax      :    768.2 us -> 7


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn các lưới nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `10 × 10`.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "bfs": rotting_bfs,
    "timestamp": rotting_timestamp,
    "per_source": rotting_per_source,
    "relax": rotting_relax,
    "simulate": rotting_simulate,
    "Solution": lambda g: Solution().orangesRotting([row[:] for row in g]),
    "SolutionBFS": lambda g: SolutionBFS().orangesRotting([row[:] for row in g]),
    "SolutionTimestamp": lambda g: SolutionTimestamp().orangesRotting([row[:] for row in g]),
    "SolutionPerSource": lambda g: SolutionPerSource().orangesRotting([row[:] for row in g]),
    "SolutionRelax": lambda g: SolutionRelax().orangesRotting([row[:] for row in g]),
    "SolutionSimulate": lambda g: SolutionSimulate().orangesRotting([row[:] for row in g]),
}


def check(grid, want=None, names=None):
    if want is None:
        want = rotting_simulate(grid)
    snapshot = [row[:] for row in grid]
    for name in (names or IMPLS):
        got = IMPLS[name](grid)
        assert got == want, (name, grid, got, want)
        assert grid == snapshot, ("cài đặt làm hỏng đầu vào", name)
    return want


for g, want in ex:
    check(g, want)
print("ví dụ đề: mô phỏng nguyên văn và", len(IMPLS), "cài đặt đúng (không sửa đầu vào)")

ví dụ đề: mô phỏng nguyên văn và 11 cài đặt đúng (không sửa đầu vào)


### B. Edge case

- Lưới `1 × 1`: `[[0]]`, `[[1]]`, `[[2]]`.
- Không có cam tươi (kể cả có cam thối hay không): đáp án `0`.
- Có cam tươi nhưng **không** có cam thối: `-1`.
- Cam tươi bị ô trống chặn hoàn toàn: `-1`.
- Cam thối ở giữa lưới, hay ở góc (chuỗi dài).
- Nhiều cam thối cùng lúc lây từ hai phía (BFS nhiều nguồn).
- Toàn cam thối; toàn cam tươi và một cam thối ở góc `10 × 10` (đáp án `18`).
- Đường đi ngoằn ngoèo (rắn).

In [19]:
snake = [[2, 1, 1, 1, 1], [0, 0, 0, 0, 1], [1, 1, 1, 1, 1], [1, 0, 0, 0, 0], [1, 1, 1, 1, 1]]
edge = [
    [[0]], [[1]], [[2]], [[0, 0]], [[1, 1]], [[2, 2]], [[0, 2]], [[2, 0]], [[2, 1]], [[1, 2]], [[1, 0, 2]], [[2, 0, 1]], [[2, 1, 1], [0, 1, 1], [1, 0, 1]],
    [[2, 1, 1], [1, 1, 0], [0, 1, 1]], [[1], [1], [2]], [[2], [1], [1]], [[2], [0], [1]], snake, [[2, 1, 1, 1, 2]], [[1, 2, 1], [2, 1, 2], [1, 2, 1]],
    [[2] + [1] * 9 for _ in range(1)], [[2] * 10 for _ in range(10)], [[1] * 10 for _ in range(10)],
]
corner = [[1] * 10 for _ in range(10)]
corner[0][0] = 2
edge.append(corner)
for g in edge:
    check(g)
assert Solution().orangesRotting([row[:] for row in corner]) == 18 == rotting_simulate(corner)
assert Solution().orangesRotting([row[:] for row in snake]) == rotting_simulate(snake)
print(len(edge), "edge case: mọi cài đặt khớp mô phỏng; góc 10×10 -> 18; rắn ->", rotting_simulate(snake))

24 edge case: mọi cài đặt khớp mô phỏng; góc 10×10 -> 18; rắn -> 16


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_off_by_one(grid):                              # đếm thừa một phút (không dừng khi hết cam tươi)
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if g[r][c] == 2)
    fresh = sum(row.count(1) for row in g)
    minutes = 0
    while queue:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for rr, cc in neighbors(r, c, m, n):
                if g[rr][cc] == 1:
                    g[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc))
        minutes += 1
    return minutes if fresh == 0 else -1


def bug_no_minus_one(grid):                            # không trả -1 khi còn cam tươi
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue = deque((r, c, 0) for r in range(m) for c in range(n) if g[r][c] == 2)
    last = 0
    while queue:
        r, c, t = queue.popleft()
        last = max(last, t)
        for rr, cc in neighbors(r, c, m, n):
            if g[rr][cc] == 1:
                g[rr][cc] = 2
                queue.append((rr, cc, t + 1))
    return last


def bug_diagonal(grid):                                # 8 hướng
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    d8 = [(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0)]
    queue = deque((r, c, 0) for r in range(m) for c in range(n) if g[r][c] == 2)
    fresh = sum(row.count(1) for row in g)
    last = 0
    while queue:
        r, c, t = queue.popleft()
        last = max(last, t)
        for dr, dc in d8:
            rr, cc = r + dr, c + dc
            if 0 <= rr < m and 0 <= cc < n and g[rr][cc] == 1:
                g[rr][cc] = 2
                fresh -= 1
                queue.append((rr, cc, t + 1))
    return last if fresh == 0 else -1


def bug_single_source(grid):                           # chỉ BFS từ cam thối ĐẦU TIÊN
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    src = next(((r, c) for r in range(m) for c in range(n) if g[r][c] == 2), None)
    fresh = sum(row.count(1) for row in g)
    if src is None:
        return 0 if fresh == 0 else -1
    queue = deque([(src[0], src[1], 0)])
    last = 0
    while queue:
        r, c, t = queue.popleft()
        last = max(last, t)
        for rr, cc in neighbors(r, c, m, n):
            if g[rr][cc] == 1:
                g[rr][cc] = 2
                fresh -= 1
                queue.append((rr, cc, t + 1))
    return last if fresh == 0 else -1


def bug_empty_means_minus1(grid):                      # không có cam tươi ban đầu mà trả -1
    r = rotting_bfs(grid)
    return -1 if not any(1 in row for row in grid) else r


def bug_same_minute_chain(grid):                       # sửa tại chỗ khi quét: cam vừa thối lây ngay trong cùng phút
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    minutes = 0
    while True:
        changed = False
        for r in range(m):
            for c in range(n):
                if g[r][c] == 1 and any(g[rr][cc] == 2 for rr, cc in neighbors(r, c, m, n)):
                    g[r][c] = 2                        # BUG: cập nhật tại chỗ
                    changed = True
        if not changed:
            break
        minutes += 1
    return -1 if any(1 in row for row in g) else minutes


def bug_no_rotten_zero(grid):                          # không có cam thối mà còn cam tươi thì trả 0
    r = rotting_bfs(grid)
    return 0 if r == -1 and not any(2 in row for row in grid) else r


def bug_count_nonzero(grid):                           # đếm cả cam thối vào "fresh"
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if g[r][c] == 2)
    fresh = sum(1 for row in g for x in row if x != 0)
    minutes = 0
    while queue and fresh:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for rr, cc in neighbors(r, c, m, n):
                if g[rr][cc] == 1:
                    g[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc))
        minutes += 1
    return minutes if fresh == 0 else -1


BUGS = [bug_off_by_one, bug_no_minus_one, bug_diagonal, bug_single_source, bug_empty_means_minus1, bug_same_minute_chain, bug_no_rotten_zero, bug_count_nonzero]
pool = [[list(r) for r in rows] for rows in itertools.product(itertools.product((0, 1, 2), repeat=3), repeat=2)]
for bug in BUGS:
    found = next((g for g in pool if bug(g) != rotting_simulate(g)), None)
    assert found is not None, bug.__name__
    print(f"{bug.__name__:24s} sai trên {found}: ra {bug(found)}, đúng {rotting_simulate(found)}")

bug_off_by_one           sai trên [[0, 0, 0], [0, 0, 2]]: ra 1, đúng 0
bug_no_minus_one         sai trên [[0, 0, 0], [0, 0, 1]]: ra 0, đúng -1
bug_diagonal             sai trên [[0, 0, 1], [0, 2, 0]]: ra 1, đúng -1
bug_single_source        sai trên [[0, 0, 0], [2, 2, 1]]: ra -1, đúng 1
bug_empty_means_minus1   sai trên [[0, 0, 0], [0, 0, 0]]: ra -1, đúng 0
bug_same_minute_chain    sai trên [[0, 0, 0], [2, 1, 1]]: ra 1, đúng 2
bug_no_rotten_zero       sai trên [[0, 0, 0], [0, 0, 1]]: ra 0, đúng -1
bug_count_nonzero        sai trên [[0, 0, 0], [0, 0, 2]]: ra -1, đúng 0


### D. Vét cạn các lưới nhỏ

Mọi lưới với các kích thước `1 × n` (`n = 1..8`), `2 × n` (`n = 1..5`), `3 × 1`, `3 × 2`, `3 × 3`, `4 × 2` (tổng `103 269` lưới, mỗi ô thuộc `{0, 1, 2}`), rồi `6 000` lưới ngẫu nhiên kích thước `3 × 4`, `4 × 3`, `4 × 4`, `5 × 5`, `2 × 7`, `6 × 3` với mật độ khác nhau. Oracle: mô phỏng từng phút đúng nguyên văn đề.

In [21]:
def all_grids(m, n):
    for cells in itertools.product((0, 1, 2), repeat=m * n):
        yield [list(cells[r * n:(r + 1) * n]) for r in range(m)]


t0 = time.perf_counter()
fast = [k for k in IMPLS if k != "simulate"]
cnt = 0
dist_of_answers = {}
for m, n in [(1, k) for k in range(1, 9)] + [(2, k) for k in range(1, 6)] + [(3, 1), (3, 2), (3, 3), (4, 2)]:
    for g in all_grids(m, n):
        want = rotting_simulate(g)
        for name in fast:
            assert IMPLS[name](g) == want, (name, g, IMPLS[name](g), want)
        cnt += 1
        dist_of_answers[want] = dist_of_answers.get(want, 0) + 1
print(cnt, "lưới nhỏ (1×1..8, 2×1..5, 3×1..3, 4×2):", len(fast), "cài đặt khớp mô phỏng nguyên văn;", round(time.perf_counter() - t0, 1), "s")
print("phân bố đáp án (-1: không thể):", dict(sorted(dist_of_answers.items())))
assert sum(3 ** (m * n) for m, n in [(1, k) for k in range(1, 9)] + [(2, k) for k in range(1, 6)] + [(3, 1), (3, 2), (3, 3), (4, 2)]) == cnt

103269 lưới nhỏ (1×1..8, 2×1..5, 3×1..3, 4×2): 10 cài đặt khớp mô phỏng nguyên văn; 11.0 s
phân bố đáp án (-1: không thể): {-1: 27787, 0: 2714, 1: 39018, 2: 25022, 3: 6142, 4: 1970, 5: 512, 6: 98, 7: 6}


In [22]:
# 3×4, 4×4 và 5×5 ngẫu nhiên (mật độ khác nhau) so với mô phỏng nguyên văn
rng = random.Random(99401)
t0 = time.perf_counter()
cnt = 0
for _ in range(6000):
    m, n = rng.choice([(3, 4), (4, 3), (4, 4), (5, 5), (2, 7), (6, 3)])
    g = random_grid(rng, m, n, rng.choice([0.0, 0.1, 0.3, 0.5]), rng.choice([0.02, 0.1, 0.3]))
    want = rotting_simulate(g)
    for name in fast:
        assert IMPLS[name](g) == want, (name, g)
    cnt += 1
print(cnt, "lưới ngẫu nhiên 3×4..6×3, 5×5: khớp mô phỏng;", round(time.perf_counter() - t0, 1), "s")

6000 lưới ngẫu nhiên 3×4..6×3, 5×5: khớp mô phỏng; 1.4 s


### E. Tính chất / ngẫu nhiên (lưới tới `10 × 10`, seed cố định)

(1) Các cách khớp nhau và khớp mô phỏng; (2) **công thức** `max_x min_s d(x, s)`: bằng `rotting_per_source` và bằng đáp án (trừ khi `-1`); (3) **đối xứng**: chuyển vị, lật ngang, lật dọc không đổi đáp án; (4) **đơn điệu**: biến một cam tươi thành cam thối không làm đáp án **tăng** (nếu cả hai khác `-1`) và không biến một đáp án hữu hạn thành `-1`; (5) `0 ≤ đáp án ≤` số cam tươi (nếu khác `-1`); (6) đáp án `0` ⇔ không có cam tươi; (7) đầu vào không bị sửa.

In [23]:
def transpose(g):
    return [list(col) for col in zip(*g)]


rng = random.Random(20240994)
t0 = time.perf_counter()
cnt_pos = cnt_neg = 0
for it in range(3000):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    g = random_grid(rng, m, n, rng.choice([0.0, 0.1, 0.25, 0.4]), rng.choice([0.01, 0.05, 0.15]))
    snap = [row[:] for row in g]
    results = {name: IMPLS[name](g) for name in fast}
    assert len(set(results.values())) == 1, (g, results)                                  # (1)
    ans = results["Solution"]
    assert ans == rotting_simulate(g)                                                     # (1)
    assert g == snap                                                                      # (7)
    assert rotting_per_source(g) == ans                                                   # (2)
    for variant in (transpose(g), [row[::-1] for row in g], g[::-1]):                     # (3)
        assert IMPLS["Solution"](variant) == ans
    fresh = sum(row.count(1) for row in g)
    if ans != -1:
        assert 0 <= ans <= fresh                                                          # (5)
        cnt_pos += 1
    else:
        cnt_neg += 1
    assert (ans == 0) == (fresh == 0)                                                     # (6)
    cells = [(r, c) for r in range(m) for c in range(n) if g[r][c] == 1]
    if cells:
        r, c = rng.choice(cells)
        g2 = [row[:] for row in g]
        g2[r][c] = 2
        a2 = IMPLS["Solution"](g2)
        if ans != -1:
            assert a2 != -1 and a2 <= ans                                                 # (4)
print("3000 lưới ngẫu nhiên (tới 10×10): 7 tính chất đúng; có đáp án:", cnt_pos, "| -1:", cnt_neg, ";", round(time.perf_counter() - t0, 1), "s")

3000 lưới ngẫu nhiên (tới 10×10): 7 tính chất đúng; có đáp án: 1225 | -1: 1775 ; 2.2 s


### F. Giới hạn `10 × 10`

Lưới đầy cam tươi với một cam thối ở góc (đáp án `18`), ở giữa (`9`/`10`), hai cam thối ở hai góc đối diện, lưới hình xoắn ốc (đường dài nhất), lưới kẹp bởi các bức tường tạo `-1`.

In [24]:
t0 = time.perf_counter()
full = lambda: [[1] * 10 for _ in range(10)]
g1 = full(); g1[0][0] = 2
g2 = full(); g2[5][5] = 2
g3 = full(); g3[0][0] = 2; g3[9][9] = 2
g4 = full(); g4[0][0] = 2; g4[0][9] = 2; g4[9][0] = 2; g4[9][9] = 2
# xoắn ốc: một đường đi dài (rắn) bằng các ô 1 xen các hàng 0
snake10 = [[0] * 10 for _ in range(10)]
for r in range(10):
    if r % 2 == 0:
        snake10[r] = [1] * 10
    else:
        snake10[r][9 if (r // 2) % 2 == 0 else 0] = 1
snake10[0][0] = 2
wall = full(); wall[0][0] = 2
for r in range(10):
    wall[r][5] = 0                                   # bức tường dọc chặn hoàn toàn
for name, g, want in [("góc", g1, 18), ("giữa", g2, None), ("hai góc đối", g3, None), ("bốn góc", g4, None), ("rắn", snake10, None), ("tường", wall, -1)]:
    r = {k: IMPLS[k](g) for k in IMPLS}
    assert len(set(r.values())) == 1, (name, r)
    assert r["simulate"] == rotting_per_source(g)
    if want is not None:
        assert r["Solution"] == want
    print(f"  {name:12s}: {r['Solution']}")
print("10×10: mọi cài đặt khớp mô phỏng và công thức max-min;", round(time.perf_counter() - t0, 2), "s")

  góc         : 18
  giữa        : 10
  hai góc đối : 9
  bốn góc     : 8
  rắn         : 54
  tường       : -1
10×10: mọi cài đặt khớp mô phỏng và công thức max-min; 0.02 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng thừa một phút

In [25]:
g = [[2, 1, 1], [1, 1, 0], [0, 1, 1]]
print("bug:", bug_off_by_one(g), "| đúng:", Solution().orangesRotting([r[:] for r in g]))
assert bug_off_by_one(g) == 5 and Solution().orangesRotting([r[:] for r in g]) == 4

bug: 5 | đúng: 4


**Vì sao sai:** nếu vòng lặp chỉ dừng khi hàng đợi rỗng, tầng cuối (đã không còn cam tươi để lây) vẫn được đếm một phút. **Đúng:** dừng khi `fresh == 0` (`while queue and fresh`), hoặc trả `minutes − 1`/dùng nhãn thời gian lớn nhất.

### Lỗi 2: quên trả `-1` khi còn cam tươi

In [26]:
g = [[2, 1, 1], [0, 1, 1], [1, 0, 1]]
print("bug:", bug_no_minus_one(g), "| đúng:", Solution().orangesRotting([r[:] for r in g]))
assert bug_no_minus_one(g) != -1 and Solution().orangesRotting([r[:] for r in g]) == -1

bug: 4 | đúng: -1


**Vì sao sai:** BFS chỉ tới được các cam liên thông; cam ở `(2,0)` bị chặn nên vẫn tươi. **Đúng:** đếm `fresh` và trả `-1` nếu `fresh > 0` sau BFS.

### Lỗi 3: dùng 8 hướng

In [27]:
g = [[2, 0], [0, 1]]
print("bug:", bug_diagonal(g), "| đúng:", Solution().orangesRotting([r[:] for r in g]))
assert bug_diagonal(g) == 1 and Solution().orangesRotting([r[:] for r in g]) == -1

bug: 1 | đúng: -1


**Vì sao sai:** đề chỉ cho lây **4 hướng** (trên/dưới/trái/phải); đường chéo không tính.

### Lỗi 4: chỉ BFS từ một cam thối

In [28]:
g = [[2, 1, 0, 1, 2]]
print("bug:", bug_single_source(g), "| đúng:", Solution().orangesRotting([r[:] for r in g]))
assert bug_single_source(g) == -1 and Solution().orangesRotting([r[:] for r in g]) == 1

bug: -1 | đúng: 1


**Vì sao sai:** mọi cam thối lây **đồng thời**; phải đưa tất cả vào hàng đợi từ đầu (BFS nhiều nguồn).

### Lỗi 5: lây liền trong cùng một phút (sửa lưới tại chỗ khi quét)

In [29]:
g = [[2, 1, 1, 1]]
print("bug:", bug_same_minute_chain(g), "| đúng:", Solution().orangesRotting([r[:] for r in g]))
assert bug_same_minute_chain(g) == 1 and Solution().orangesRotting([r[:] for r in g]) == 3

bug: 1 | đúng: 3


**Vì sao sai:** khi quét từ trái sang phải và sửa lưới tại chỗ, cam vừa thối lập tức lây cho ô kế tiếp trong cùng lượt quét. **Đúng:** tính trạng thái phút sau dựa trên trạng thái **đầu phút** (bản sao hoặc BFS theo tầng).

### Lỗi 6 (bonus): không có cam tươi ⇒ phải là `0`

In [30]:
print("bug:", bug_empty_means_minus1([[0, 2]]), bug_no_rotten_zero([[1]]), "| đúng:", Solution().orangesRotting([[0, 2]]), Solution().orangesRotting([[1]]))
assert bug_empty_means_minus1([[0, 2]]) == -1 and bug_no_rotten_zero([[1]]) == 0
assert Solution().orangesRotting([[0, 2]]) == 0 and Solution().orangesRotting([[1]]) == -1

bug: -1 0 | đúng: 0 -1


**Vì sao sai:** không còn cam tươi ⇒ đã đạt mục tiêu ở phút `0`; còn cam tươi mà không có cam thối ⇒ không bao giờ thối ⇒ `-1`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "lan truyền đồng thời từ nhiều nguồn theo từng bước thời gian" → **BFS nhiều nguồn**: đưa mọi nguồn vào hàng đợi trước, mỗi tầng là một đơn vị thời gian.

**Mẹo:**

1. Đếm `fresh` để quyết định `-1` và để dừng đúng lúc (tránh cộng thừa).
2. Đánh dấu ô khi **thêm vào** hàng đợi (không phải khi lấy ra).
3. Xử lý đúng `len(queue)` phần tử mỗi tầng.
4. 4 hướng, kiểm biên.
5. Đáp án = `max` khoảng cách ngắn nhất từ tập nguồn; mô hình "max của min".
6. Oracle: mô phỏng từng phút đúng nguyên văn.

**Họ bài cùng pattern:** 01 Matrix (542), Walls and Gates (286, Premium), As Far from Land as Possible (1162), Shortest Path in Binary Matrix (1091), Map of Highest Peak (1765), Pacific Atlantic Water Flow (417).

## 15. Biến thể

1. **Lan theo hướng tuỳ ý** (8 hướng, hoặc kiểu "quân mã"): tham số hoá `DIRS`.
2. **Trả về lưới thời điểm thối** (`-1` cho cam không bao giờ thối, `0` cho cam thối ban đầu, `None`/`-1` cho ô trống).
3. **Số cam thối sau mỗi phút** (dòng thời gian).

Cài đặt và đối chiếu mô phỏng:

In [31]:
def rotting_times(grid, dirs=DIRS):
    """Biến thể 2: lưới thời điểm thối. Ô trống -> None; cam thối ban đầu -> 0; cam không thối được -> -1."""
    m, n = len(grid), len(grid[0])
    t = [[None if grid[r][c] == 0 else (0 if grid[r][c] == 2 else -1) for c in range(n)] for r in range(m)]
    queue = deque((r, c) for r in range(m) for c in range(n) if grid[r][c] == 2)
    while queue:
        r, c = queue.popleft()
        for dr, dc in dirs:
            rr, cc = r + dr, c + dc
            if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] == 1 and t[rr][cc] == -1:
                t[rr][cc] = t[r][c] + 1
                queue.append((rr, cc))
    return t


def timeline(grid):
    """Biến thể 3: số cam thối sau phút 0, 1, 2, ... cho tới khi không đổi."""
    g = [row[:] for row in grid]
    out = [sum(row.count(2) for row in g)]
    while True:
        new = one_minute(g)
        if new == g:
            return out
        g = new
        out.append(sum(row.count(2) for row in g))


def simulate_dirs(grid, dirs):
    """Mô phỏng nguyên văn với tập hướng tuỳ ý (oracle cho biến thể 1)."""
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    minutes = 0
    while True:
        new = [row[:] for row in g]
        for r in range(m):
            for c in range(n):
                if g[r][c] == 1 and any(0 <= r + dr < m and 0 <= c + dc < n and g[r + dr][c + dc] == 2 for dr, dc in dirs):
                    new[r][c] = 2
        if new == g:
            break
        g, minutes = new, minutes + 1
    return -1 if any(1 in row for row in g) else minutes


D8 = [(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0)]
KNIGHT = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
rng = random.Random(15)
for _ in range(600):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = random_grid(rng, m, n, rng.choice([0.0, 0.2]), rng.choice([0.05, 0.2]))
    t = rotting_times(g)
    worst = max([x for row in t for x in row if x is not None] + [0])
    ans = rotting_simulate(g)
    assert (ans == -1) == any(x == -1 for row in t for x in row)                         # (2)
    if ans != -1:
        assert worst == ans
    tl = timeline(g)
    if ans != -1:
        assert len(tl) - 1 == ans and tl[-1] == sum(row.count(1) + row.count(2) for row in g)   # (3)
    for dirs in (D8, KNIGHT):                                                             # (1)
        tt = rotting_times(g, dirs)
        w = max([x for row in tt for x in row if x is not None] + [0])
        a = simulate_dirs(g, dirs)
        assert (a == -1) == any(x == -1 for row in tt for x in row)
        if a != -1:
            assert w == a
print("lưới thời điểm thối, dòng thời gian, hướng tuỳ ý (8 hướng, quân mã): khớp mô phỏng")

lưới thời điểm thối, dòng thời gian, hướng tuỳ ý (8 hướng, quân mã): khớp mô phỏng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng phút | đúng định nghĩa | `O((mn)²)` | `O(mn)` | oracle |
| 2. BFS nhiều nguồn theo tầng | tầng = phút | `O(mn)` | `O(mn)` | **đáp án chính** |
| 3. BFS nhãn thời gian | `(r, c, t)` | `O(mn)` | `O(mn)` | không cần vòng tầng |
| 4. BFS từng nguồn + min | `max_x min_s d` | `O(R·mn)` | `O(mn)` | minh hoạ công thức |
| 5. Nới lỏng | Bellman–Ford | `O((mn)²)` | `O(mn)` | khi đồ thị có trọng số/khó BFS |

Mọi cách khớp oracle mô phỏng trên toàn bộ lưới nhỏ (mục 12D) và hàng nghìn lưới tới `10 × 10`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print(Solution().orangesRotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]), Solution().orangesRotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]),
      Solution().orangesRotting([[0, 2]]), Solution().orangesRotting([[1]]), Solution().orangesRotting([[2, 1, 1, 1, 2]]))

4 -1 0 -1 2


### Bài 2 (tính tay)

`grid = [[2,1,0,1],[1,1,0,1],[0,1,2,1]]`. (a) Các cam thối ban đầu? (b) Viết thời điểm thối của từng cam tươi. (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện BFS nhiều nguồn:

In [33]:
def bfs_todo(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue, fresh = deque(), 0
    for r in range(m):
        for c in range(n):
            pass        # TODO 1: nếu g[r][c] == 2 đưa (r, c) vào queue; nếu == 1 thì fresh += 1
    minutes = 0
    while queue and fresh:
        pass            # TODO 2: xử lý đúng len(queue) phần tử (một tầng), làm thối cam tươi kề, giảm fresh, thêm vào queue
        pass            # TODO 3: minutes += 1
        break           # xoá dòng này khi làm xong
    return -1           # TODO 4: return minutes if fresh == 0 else -1


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(bfs_todo(g) == w for g, w in ex)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **BFS gắn nhãn thời gian** `(r, c, t)`:

In [34]:
class MySolution:
    def orangesRotting(self, grid):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().orangesRotting([r[:] for r in g]) == w for g, w in ex)

### Bài 5 (mini challenge): 01 Matrix (LeetCode 542)

Cho ma trận `mat` gồm `0` và `1`, trả về ma trận `dist` với `dist[r][c]` = khoảng cách (4 hướng, bước 1) từ ô `(r, c)` tới ô `0` **gần nhất**. Dùng BFS nhiều nguồn từ mọi ô `0`. Kiểm bằng brute `O((mn)²)` (tính khoảng cách Manhattan nhỏ nhất tới mọi ô `0`).

In [35]:
def update_matrix(mat):
    pass            # TODO


# Khi làm xong:
# assert update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1 và 2 đã cho: `4`, `-1`.
- Mức 2: `[[2,1,1,1,2]]`: hai cam thối lây vào giữa; `[[1]]` không có cam thối.
- Mức 3: `4 -1 0 -1 2`.

**Bài 2**

- Mức 1: cam thối tại `(0,0)` và `(2,2)`.
- Mức 2: ô `(0,1)`, `(1,0)` thối ở phút 1; `(1,1)` ở phút 2; `(2,1)` kề `(2,2)` ở phút 1; `(2,3)` kề `(2,2)` ở phút 1; `(1,3)` ở phút 2; `(0,3)` ở phút 3.
- Mức 3: đáp án `3` (cam `(0,3)` thối cuối cùng).

**Bài 3**

- Mức 1: `queue.append((r, c))` khi `g[r][c] == 2`.
- Mức 2: `for _ in range(len(queue)):` lấy ô, duyệt 4 hướng, `g[rr][cc] == 1` ⇒ `= 2`, `fresh -= 1`, `append`.
- Mức 3: `minutes += 1` sau mỗi tầng; `return minutes if fresh == 0 else -1`.

**Bài 4**

- Mức 1: `queue = deque((r, c, 0) for ... if g[r][c] == 2)`.
- Mức 2: lấy `(r, c, t)`, `last = max(last, t)`, đẩy `(rr, cc, t + 1)`.
- Mức 3: `return last if fresh == 0 else -1`.

**Bài 5**

- Mức 1: nguồn là mọi ô `0`; các ô `1` chưa biết (`-1`).
- Mức 2: BFS: ô kề chưa thăm nhận `dist + 1`.
- Mức 3: trả ma trận khoảng cách.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = (Solution().orangesRotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]), Solution().orangesRotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]),
        Solution().orangesRotting([[0, 2]]), Solution().orangesRotting([[1]]), Solution().orangesRotting([[2, 1, 1, 1, 2]]))
print(vals)
assert vals == (4, -1, 0, -1, 2)
print("Bài 1 OK")

(4, -1, 0, -1, 2)
Bài 1 OK


### Bài 2

In [37]:
g = [[2, 1, 0, 1], [1, 1, 0, 1], [0, 1, 2, 1]]
t = rotting_times(g)
print(t)
assert t == [[0, 1, None, 3], [1, 2, None, 2], [None, 1, 0, 1]]
assert Solution().orangesRotting([r[:] for r in g]) == 3 == rotting_simulate(g)
print("Bài 2 OK: đáp án 3")

[[0, 1, None, 3], [1, 2, None, 2], [None, 1, 0, 1]]
Bài 2 OK: đáp án 3


### Bài 3

In [38]:
def bfs_answer(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    queue, fresh = deque(), 0
    for r in range(m):
        for c in range(n):
            if g[r][c] == 2:
                queue.append((r, c))
            elif g[r][c] == 1:
                fresh += 1
    minutes = 0
    while queue and fresh:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for rr, cc in neighbors(r, c, m, n):
                if g[rr][cc] == 1:
                    g[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc))
        minutes += 1
    return minutes if fresh == 0 else -1


assert all(bfs_answer(g) == w for g, w in ex)
for g in itertools.islice(all_grids(3, 3), 0, 19683, 7):
    assert bfs_answer(g) == rotting_simulate(g)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def orangesRotting(self, grid):
        m, n = len(grid), len(grid[0])
        queue = deque((r, c, 0) for r in range(m) for c in range(n) if grid[r][c] == 2)
        fresh = sum(row.count(1) for row in grid)
        last = 0
        while queue:
            r, c, t = queue.popleft()
            last = max(last, t)
            for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                rr, cc = r + dr, c + dc
                if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] == 1:
                    grid[rr][cc] = 2
                    fresh -= 1
                    queue.append((rr, cc, t + 1))
        return last if fresh == 0 else -1


assert all(MySolutionAnswer().orangesRotting([r[:] for r in g]) == w for g, w in ex)
for g in itertools.islice(all_grids(3, 3), 0, 19683, 7):
    assert MySolutionAnswer().orangesRotting([r[:] for r in g]) == rotting_simulate(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def update_matrix(mat):
    m, n = len(mat), len(mat[0])
    dist = [[-1] * n for _ in range(m)]
    queue = deque()
    for r in range(m):
        for c in range(n):
            if mat[r][c] == 0:
                dist[r][c] = 0
                queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        for rr, cc in neighbors(r, c, m, n):
            if dist[rr][cc] == -1:
                dist[rr][cc] = dist[r][c] + 1
                queue.append((rr, cc))
    return dist


def update_matrix_brute(mat):
    m, n = len(mat), len(mat[0])
    zeros = [(r, c) for r in range(m) for c in range(n) if mat[r][c] == 0]
    return [[min((abs(r - a) + abs(c - b) for a, b in zeros), default=-1) for c in range(n)] for r in range(m)]


assert update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
rng = random.Random(5)
for _ in range(500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[rng.choice([0, 1, 1]) for _ in range(n)] for _ in range(m)]
    if any(0 in row for row in mat):
        assert update_matrix(mat) == update_matrix_brute(mat)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (lan truyền đồng thời từ nhiều nguồn theo thời gian → BFS nhiều nguồn theo tầng):**

```python
class Solution:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        queue, fresh = deque(), 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 2:
                    queue.append((r, c))
                elif grid[r][c] == 1:
                    fresh += 1
        minutes = 0
        while queue and fresh:
            for _ in range(len(queue)):
                r, c = queue.popleft()
                for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < m and 0 <= cc < n and grid[rr][cc] == 1:
                        grid[rr][cc] = 2
                        fresh -= 1
                        queue.append((rr, cc))
            minutes += 1
        return minutes if fresh == 0 else -1
```

**Checklist trước khi submit:**

- [ ] Đưa **mọi** cam thối vào hàng đợi từ đầu (BFS nhiều nguồn)?
- [ ] Đếm `fresh` và trả `-1` khi còn sót; trả `0` khi không có cam tươi?
- [ ] Mỗi tầng = một phút; vòng ngoài `while queue and fresh` (tránh cộng thừa)?
- [ ] Đánh dấu ô thối **ngay khi thêm** vào hàng đợi?
- [ ] Chỉ 4 hướng; kiểm biên?
- [ ] Đã thử ví dụ 1, 2, 3, `[[1]]`, `[[0]]`, góc `10×10` (`18`)?
- [ ] Đúng tên `class Solution`, `orangesRotting(self, grid)`, không in debug.